# 07 — Score live flights + alternative-flight recommender

Applies the fitted feature pipeline from `04_gold` to `api_silver_flights`, loads the
champion models from Unity Catalog **by alias**, scores both variants at each model's own
tuned threshold, and upserts into:

- `flight_delay_predictions` — one row per flight per scoring run
- `alternative_flight_recommendations` — up to 5 lower-risk alternatives per flight,
  same route, ±3 hours, at least 10 points better

This notebook is the other half of the loop the original project never closed. `05_train`
registers under a 3-level UC name with a `@champion` alias; this one resolves that alias
and never mentions a run ID or a version number.


In [0]:
import sys
sys.path.append("..")

import mlflow
from mlflow.tracking import MlflowClient
from delta.tables import DeltaTable
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array
from pyspark.ml.feature import VectorSlicer
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from src import config

mlflow.set_registry_uri(config.MLFLOW_REGISTRY_URI)
client = MlflowClient()
print(f"Registry: {mlflow.get_registry_uri()}")


Registry: databricks-uc


## Resolve the champions

`05_train` registers **one** champion per variant — whichever of RF or GBT won, after the
tie rule. Which name that is depends on the data, so this notebook cannot assume it. It
asks the registry which of the candidate names currently carries the `@champion` alias.

The decision threshold travels with the model as a version tag. Reading it here rather
than hardcoding a number is what makes retraining a promotion rather than a code change:
if the next run picks a different cut, this notebook follows it without being edited.


In [0]:
def load_champion(candidates, variant):
    """Return (model, threshold, name, version) for the current champion."""
    errors, found = {}, []
    for name in candidates:
        try:
            mv = client.get_model_version_by_alias(name, config.CHAMPION_ALIAS)
        except Exception as e:
            errors[name] = type(e).__name__
            continue

        threshold = mv.tags.get("decision_threshold")
        if threshold is None:
            # Fallback for a model registered before the tag existed.
            try:
                threshold = client.get_run(mv.run_id).data.tags.get("decision_threshold")
            except Exception:
                threshold = None
        if threshold is None:
            raise ValueError(
                f"{name} v{mv.version} carries no decision_threshold tag. Re-run 05_train; "
                "scoring at Spark's default 0.5 produced zero positive predictions for the "
                "pre-departure model."
            )

        found.append((name, mv, float(threshold)))

    if not found:
        raise RuntimeError(
            f"No @{config.CHAMPION_ALIAS} alias on any of {candidates} ({errors}). "
            "Run 05_train first."
        )

    # More than one name carrying @champion means a previous run's alias was never
    # cleared. 05_train removes the loser's alias now, but a registry that predates
    # that fix still has both. Take the most recently created version and say so
    # loudly rather than silently serving whichever was tried first.
    if len(found) > 1:
        print(f"  WARNING: {len(found)} models carry @{config.CHAMPION_ALIAS} for "
              f"{variant}: {[n for n, _, _ in found]}")
        print(f"  Taking the most recent. Re-run 05_train to clear stale aliases.")
    found.sort(key=lambda t: int(t[1].version), reverse=True)
    found.sort(key=lambda t: t[1].creation_timestamp, reverse=True)

    name, mv, threshold = found[0]
    model = mlflow.spark.load_model(
        f"models:/{name}@{config.CHAMPION_ALIAS}", dfs_tmpdir=config.ARTIFACT_VOLUME
    )
    print(f"{variant:<14} {name}  v{mv.version}  threshold={threshold:.2f}")
    return model, threshold, name, mv.version


pre_model, PRE_THRESHOLD, pre_name, pre_version = load_champion(
    [config.MODEL_GBT_PRE, config.MODEL_RF_PRE], "pre-departure"
)
in_model, IN_THRESHOLD, in_name, in_version = load_champion(
    [config.MODEL_GBT_IN, config.MODEL_RF_IN], "in-flight"
)


{"ts": "2026-09-13 02:39:28.868", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTQ4NTY4MzM3ODEwNDQyNRABIAEyJDAxYTA5OGExLTQwNDUtNzQ0OS04MmViLTVjOTg0NjcyYTE3MzokMzNlOWQyMzctNjc2OC0zZjY3LTllMzQtZDJkYzcyZTkwOTI5SgwIm5mY1QYQwJry1QJQAVgBYAFoxoPV4f+t4QE=.", "context": {}}
{"ts": "2026-09-13 02:39:28.868", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTQ4NTY4MzM3ODEwNDQyNRABIAEyJDAxYTA5OGExLTQwNDUtNzQ0OS04MmViLTVjOTg0NjcyYTE3MzokMzNlOWQyMzctNjc2OC0zZjY3LTllMzQtZDJkYzcyZTkwOTI5SgwIm5mY1QYQwJry1QJQAVgBYAFoxoPV4f+t4QE=.", "context": {}}
{"ts": "2026-09-13 02:39:28.868", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTQ4NTY4MzM3ODEwNDQyNRABIAEyJDAxYTA5OGExLTQwNDUtNzQ0OS04MmViLTVjOTg0NjcyYTE3MzokMzNlOWQyMzctNjc2OC0zZjY3LTllMzQtZDJkYzcyZTkwOTI5

pre-departure  workspace.flights.rf_pre_departure  v6  threshold=0.17


in-flight      workspace.flights.rf_in_flight  v4  threshold=0.38


## Feature space — from the manifest, not from a magic number

The pre-departure model is trained without `dep_delay`, so scoring has to remove the same
slot. This used to be `[i for i in range(n) if i != 11]`, which is the defect `05_train`
had removed and this notebook had quietly kept: reorder `numerical_cols` in `04_gold` and
the pre-departure model starts scoring *with* departure delay, which looks like a
suspiciously good prediction rather than an error.

The manifest `04_gold` writes is the same source `05_train` reads, so both sides of the
loop agree by construction.


In [0]:
feature_pipeline = PipelineModel.load(f"{config.ARTIFACT_VOLUME}/feature_pipeline")
api_silver = spark.table(config.API_SILVER)
print(f"Rows to score: {api_silver.count():,}")

manifest = spark.table(config.FEATURE_MANIFEST).orderBy("vector_index").collect()
index_by_name = {r["name"]: int(r["vector_index"]) for r in manifest}
n_features = len(index_by_name)

dep_delay_idx = index_by_name["dep_delay"]
pre_indices = [i for i in range(n_features) if i != dep_delay_idx]
assert dep_delay_idx not in pre_indices
print(f"Vector width {n_features}; dep_delay at {dep_delay_idx} (from the manifest)")


Rows to score: 62
Vector width 818; dep_delay at 11 (from the manifest)


In [0]:
# Fill only the columns the pipeline actually consumes, matching 04_gold's treatment.
numeric_like = [f.name for f in api_silver.schema.fields
                if f.dataType.typeName() in ("double", "integer", "long", "float")]
string_like = [f.name for f in api_silver.schema.fields if f.dataType.typeName() == "string"]

api_prepared = (
    api_silver
    .withColumn("dep_hour", (F.col("crs_dep_time") / 100).cast("int"))
    .withColumn("arr_hour", (F.col("crs_arr_time") / 100).cast("int"))
    .na.fill(0, subset=numeric_like)
    .na.fill("UNKNOWN", subset=string_like)
)

# Scoring-time feature parity.
#
# 04_gold's pipeline now expects the congestion features 03_silver builds, and
# api_silver_flights has none of them — this is what FIELD_NOT_FOUND on
# `sched_deps_origin_hour` was. Whatever the model was trained on has to exist
# here under the same names, or the transform cannot run.
#
# The two the model actually uses are reconstructed below. They are handled
# differently because they fail differently on a partial feed:
#
#   schedule_padding      needs a route median, which is a stable property of the
#                         schedule. Taken from Silver, where it is computed over
#                         four years rather than over whatever the API returned.
#
#   dep_sequence_in_day   is an ordinal within a carrier's departures from an
#                         airport that day. The API returns one route, not a full
#                         airport-day, so the sequence is computed over the flights
#                         actually fetched and is an under-count of the true
#                         position. Documented in docs/API_STRATEGY.md: the fix is
#                         to pull schedules per airport-day, which is the ingestion
#                         shape the congestion features require anyway.
route_medians = (
    spark.table(config.SILVER)
    .groupBy("origin_airport_code", "destination_airport_code")
    .agg(F.expr("percentile_approx(crs_elapsed_time, 0.5)").alias("route_median_elapsed"))
)
global_median = (
    spark.table(config.SILVER)
    .agg(F.expr("percentile_approx(crs_elapsed_time, 0.5)")).first()[0]
)

carrier_day = (
    Window.partitionBy("origin_airport_code", "airline_code", "flight_date")
    .orderBy("dep_minutes")
)

# Rebuild the congestion features that 03_silver creates and the pipeline expects.
origin_hour_win = Window.partitionBy(
    "origin_airport_code", "flight_date", "dep_hour"
)
bank_win = (
    Window.partitionBy("origin_airport_code", "flight_date")
    .orderBy("dep_minutes")
    .rangeBetween(-60, 60)
)

api_prepared = (
    api_prepared
    .withColumn("dep_minutes",
                (F.floor(F.col("crs_dep_time") / 100) * 60
                 + (F.col("crs_dep_time") % 100)).cast("int"))
    .join(F.broadcast(route_medians),
          on=["origin_airport_code", "destination_airport_code"], how="left")
    .withColumn("route_median_elapsed",
                F.coalesce(F.col("route_median_elapsed"), F.lit(global_median)))
    .withColumn("schedule_padding",
                F.col("crs_elapsed_time") - F.col("route_median_elapsed"))
    .withColumn("dep_sequence_in_day", F.row_number().over(carrier_day))
    .withColumn("sched_deps_origin_hour", F.count("*").over(origin_hour_win))
    .withColumn("dep_bank_density", F.count("*").over(bank_win))
    .drop("route_median_elapsed")
)

unmatched = api_prepared.filter(F.col("schedule_padding").isNull()).count()
print(f"Scoring-time features rebuilt. Rows with no route median: {unmatched}")
print(f"  (those fall back to the global median of {global_median:.0f} min)")

# Fail here, with the missing names, rather than inside the pipeline transform.
expected = set(spark.table(config.FEATURE_MANIFEST).select("source_column")
               .distinct().toPandas()["source_column"])
base_cols = {c for c in expected if not c.endswith("_ohe")}
missing = sorted(base_cols - set(api_prepared.columns))
if missing:
    raise ValueError(
        f"api_silver_flights is missing {missing}, which 04_gold's pipeline expects. "
        "Either 03_silver added features that scoring does not rebuild, or the feature "
        "pipeline is newer than the API projection in 06_api_ingest."
    )
print(f"All {len(base_cols)} pipeline input columns present.")

transformed = feature_pipeline.transform(api_prepared)
scored_input = VectorSlicer(
    inputCol="features", outputCol="features_pre", indices=pre_indices
).transform(transformed)


Scoring-time features rebuilt. Rows with no route median: 0
  (those fall back to the global median of 126 min)
All 20 pipeline input columns present.


## Score both variants

Each champion is a `PipelineModel` containing its feature selector *and* its classifier,
and is applied whole. The previous version pulled `model.stages[0]` out and called it "the
classifier" — which after the `05_train` rewrite is the selector, and which in any case
applied a different feature space at scoring than the model was trained on. Logging the
selector and classifier as one artifact only helps if scoring uses the artifact.

Because both models expect their features in a column called `features`, the pre-departure
view is renamed into place rather than the model being reconfigured.


In [0]:
def score_variant(df, model, features_col, threshold, prefix):
    """Apply a champion pipeline whole and emit probability + decision at its threshold."""
    renamed = df.withColumn("_orig_features", F.col("features")).drop("features") \
                .withColumnRenamed(features_col, "features")

    out = model.transform(renamed)
    out = (
        out.withColumn(f"prob_{prefix}", vector_to_array("probability")[1])
           .withColumn(f"pred_{prefix}",
                       (F.col(f"prob_{prefix}") >= F.lit(threshold)).cast("int"))
           .drop("rawPrediction", "probability", "prediction", "selected", "features")
           .withColumnRenamed("_orig_features", "features")
    )
    return out


scored = score_variant(scored_input, pre_model, "features_pre", PRE_THRESHOLD, "pre")
scored = score_variant(
    scored.withColumn("features_all", F.col("features")),
    in_model, "features_all", IN_THRESHOLD, "in",
)

# Risk bands are anchored on each model's own threshold rather than on 0.5/0.7.
# The pre-departure cut is well below 0.5, so fixed bands put every flight in "Low"
# and the table would say nothing.
def risk_band(prob_col, threshold):
    return (
        F.when(F.col(prob_col) >= threshold * 1.5, F.lit("High"))
         .when(F.col(prob_col) >= threshold, F.lit("Medium"))
         .otherwise(F.lit("Low"))
    )


scored = (
    scored
    .withColumn("risk_pre", risk_band("prob_pre", PRE_THRESHOLD))
    .withColumn("risk_in", risk_band("prob_in", IN_THRESHOLD))
)
print(f"Bands — pre-departure: Medium >= {PRE_THRESHOLD:.2f}, High >= {PRE_THRESHOLD * 1.5:.2f}")
print(f"        in-flight    : Medium >= {IN_THRESHOLD:.2f}, High >= {IN_THRESHOLD * 1.5:.2f}")


Bands — pre-departure: Medium >= 0.17, High >= 0.26
        in-flight    : Medium >= 0.38, High >= 0.57


### Routing each flight to the model that fits its phase

`06_api_ingest` attaches `flight_phase` from OpenSky's `on_ground` flag. Both models are still
applied to every row — scoring is cheap and the comparison is informative — but
`recommended_model` records which one is *valid* for each flight, and the summary reports the
split.

This matters because the in-flight model is only meaningful once `dep_delay` exists. Serving
its output for a flight that has not left the gate means serving a prediction built on a
feature whose value is not yet known, which is a leak at inference time rather than at
training time.


In [0]:
# Phase comes from OpenSky via 06_api_ingest. Older API Silver tables predate the
# column, so fall back rather than fail.
if "flight_phase" not in scored.columns:
    scored = scored.withColumn("flight_phase", F.lit("unknown"))
    print("No flight_phase column — re-run 06_api_ingest with USE_OPENSKY=true.")

scored = scored.withColumn(
    "recommended_model",
    F.when(
        (F.col("flight_phase") == "airborne") & F.col("dep_delay").isNotNull(),
        F.lit("in_flight"),
    ).otherwise(F.lit("pre_departure")),
).withColumn(
    "recommended_prob_pct",
    F.when(F.col("recommended_model") == "in_flight", F.col("prob_in") * 100)
     .otherwise(F.col("prob_pre") * 100),
)

display(
    scored.groupBy("flight_phase", "recommended_model")
    .agg(F.count("*").alias("flights"),
         F.round(F.avg("recommended_prob_pct"), 2).alias("avg_delay_prob_pct"))
    .orderBy("flight_phase")
)
print("`unknown` falls back to pre-departure: it is the variant that does not")
print("require a departure to have already happened, so it is the safe default.")


No flight_phase column — re-run 06_api_ingest with USE_OPENSKY=true.


flight_phase,recommended_model,flights,avg_delay_prob_pct
unknown,pre_departure,62,20.02


`unknown` falls back to pre-departure: it is the variant that does not
require a departure to have already happened, so it is the safe default.


## Predictions — MERGE, not overwrite

The previous version wrote `mode("overwrite")`, so every run destroyed the last run's
predictions. For a table that is supposed to represent live scoring that is the wrong
semantics twice over: there is no record of what was predicted before the flight departed,
which is exactly the record you need to evaluate the model later.

A `MERGE` on (flight, date, scoring run) makes the job **idempotent** — re-running after a
failure updates in place instead of duplicating — and keeps history across runs. This is
the pattern the medallion architecture exists to enable, and it is the one thing this
pipeline was not using Delta for.


In [0]:
# Verify scored DataFrame has valid schema before proceeding
try:
    scored_cols = scored.columns
    if not scored_cols or len(scored_cols) == 0:
        raise RuntimeError(
            "UPSTREAM ERROR in cells 7-9: The feature pipeline transformation failed.\n\n"
            "The stored feature pipeline (from notebook 04_gold) expects columns that don't exist "
            "in the current api_silver table.\n\n"
            "FIX: Run notebook 04_gold to regenerate the feature pipeline, then run 05_train to "
            "retrain the models with the updated pipeline. After that, this scoring notebook will work."
        )
except Exception as e:
    error_msg = str(e)
    if "FIELD_NOT_FOUND" in error_msg or "sched_deps_origin_hour" in error_msg:
        raise RuntimeError(
            f"UPSTREAM ERROR in cells 7-9: Feature pipeline schema mismatch.\n\n"
            f"The stored feature pipeline (from notebook 04_gold) was trained on data with columns "
            f"that no longer exist in api_silver (e.g., 'sched_deps_origin_hour').\n\n"
            f"FIX: Run notebook 04_gold to regenerate the feature pipeline with the current schema, "
            f"then run 05_train to retrain the models.\n\n"
            f"Original error: {error_msg}"
        ) from e
    raise

predictions = scored.select(
    F.current_timestamp().alias("prediction_timestamp"),
    F.current_date().alias("scoring_date"),
    "airline_name", "airline_code", "fl_number",
    F.concat_ws(" -> ", "origin_airport_code", "destination_airport_code").alias("route"),
    "origin_airport_code", "destination_airport_code",
    "flight_date", "crs_dep_time", "crs_arr_time", "dep_delay",
    (F.col("prob_pre") * 100).alias("prob_delay_pre_pct"),
    F.col("pred_pre").alias("predicted_delayed_pre"),
    F.col("risk_pre").alias("risk_pre_departure"),
    (F.col("prob_in") * 100).alias("prob_delay_in_pct"),
    F.col("pred_in").alias("predicted_delayed_in"),
    F.col("risk_in").alias("risk_in_flight"),
    F.lit(pre_name).alias("model_pre"),
    F.lit(str(pre_version)).alias("model_pre_version"),
    F.lit(PRE_THRESHOLD).alias("threshold_pre"),
    F.lit(in_name).alias("model_in"),
    F.lit(str(in_version)).alias("model_in_version"),
    F.lit(IN_THRESHOLD).alias("threshold_in"),
)

# Define the expected columns explicitly to avoid triggering lazy schema resolution
expected_cols = [
    "prediction_timestamp", "scoring_date", "airline_name", "airline_code", "fl_number",
    "route", "origin_airport_code", "destination_airport_code", "flight_date",
    "crs_dep_time", "crs_arr_time", "dep_delay", "prob_delay_pre_pct",
    "predicted_delayed_pre", "risk_pre_departure", "prob_delay_in_pct",
    "predicted_delayed_in", "risk_in_flight", "model_pre", "model_pre_version",
    "threshold_pre", "model_in", "model_in_version", "threshold_in"
]

if spark.catalog.tableExists(config.PREDICTIONS):
    existing_cols = set(spark.table(config.PREDICTIONS).columns)
    source_cols = set(expected_cols)
    missing = source_cols - existing_cols
    if missing:
        spark.sql(f"ALTER TABLE {config.PREDICTIONS} ADD COLUMNS ({', '.join(
            f'{c} {"DATE" if c == "scoring_date" else "INT" if c.startswith("predicted_") else "DOUBLE" if c.startswith("threshold_") else "STRING"}'
            for c in expected_cols if c in missing
        )})")
        print(f"Added columns to {config.PREDICTIONS}: {sorted(missing)}")

# Deduplicate predictions before MERGE to avoid multiple source rows matching the same target.
# Keep the most recent prediction_timestamp for each (airline_code, fl_number, flight_date, scoring_date).
from pyspark.sql import Window

dedup_window = Window.partitionBy(
    "airline_code", "fl_number", "flight_date", "scoring_date"
).orderBy(F.desc("prediction_timestamp"))

predictions_deduped = (
    predictions
    .withColumn("_row_num", F.row_number().over(dedup_window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

dup_count = predictions.count() - predictions_deduped.count()
if dup_count > 0:
    print(f"⚠️  Removed {dup_count:,} duplicate prediction rows before MERGE")

if not spark.catalog.tableExists(config.PREDICTIONS):
    (predictions_deduped.limit(0).write.format("delta").mode("overwrite")
     .option("overwriteSchema", "true").saveAsTable(config.PREDICTIONS))
    print(f"Created {config.PREDICTIONS}")

(
    DeltaTable.forName(spark, config.PREDICTIONS).alias("t")
    .merge(
        predictions_deduped.alias("s"),
        "t.airline_code = s.airline_code AND t.fl_number = s.fl_number "
        "AND t.flight_date = s.flight_date AND t.scoring_date = s.scoring_date",
    )
    .whenMatchedUpdate(set={c: F.col(f"s.{c}") for c in expected_cols})
    .whenNotMatchedInsert(values={c: F.col(f"s.{c}") for c in expected_cols})
    .execute()
)

total = spark.table(config.PREDICTIONS).count()
print(f"MERGE complete. {config.PREDICTIONS} now holds {total:,} rows "
      f"across {spark.table(config.PREDICTIONS).select('scoring_date').distinct().count()} "
      f"scoring date(s).")
display(spark.sql(f"DESCRIBE HISTORY {config.PREDICTIONS}")
        .select("version", "operation", "operationMetrics").limit(5))


⚠️  Removed 8 duplicate prediction rows before MERGE
MERGE complete. workspace.flights.flight_delay_predictions now holds 170 rows across 3 scoring date(s).


version,operation,operationMetrics
4,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 8880, numTargetBytesRemoved -> 8860, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 60, executionTimeMs -> 8049, materializeSourceTimeMs -> 3236, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1429, numTargetRowsUpdated -> 60, numOutputRows -> 60, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 54, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3298)"
3,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 8860, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 6492, materializeSourceTimeMs -> 3202, numTargetRowsInserted -> 60, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1469, numTargetRowsUpdated -> 0, numOutputRows -> 60, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 0, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1695)"
2,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 3, numTargetBytesAdded -> 23288, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 9713, materializeSourceTimeMs -> 4121, numTargetRowsInserted -> 56, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2142, numTargetRowsUpdated -> 0, numOutputRows -> 56, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 56, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3353)"
1,ADD COLUMNS,Map()
0,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 54, numOutputBytes -> 7265)"


## Alternative-flight recommender

Same origin and destination, same day, within ±3 hours, at least 10 points lower delay
probability. Ranked by improvement plus a bonus for landing in a lower risk band, top 5
per flight.

**On the time window.** The previous version compared raw `HHMM` integers and called a
difference of 300 "three hours". `HHMM` is not linear in time — 13:00 minus 12:59 is 41 in
that arithmetic, and one minute in reality. Departure times are converted to minutes since
midnight first, which is what makes the window mean what it says.


In [0]:
def hhmm_to_minutes(c):
    return (F.floor(c / 100) * 60 + (c % 100)).cast("int")


scored_pool = predictions.withColumn("dep_minutes", hhmm_to_minutes(F.col("crs_dep_time")))

candidates = scored_pool.selectExpr(
    "airline_name AS alt_airline", "airline_code AS alt_airline_code",
    "fl_number AS alt_flight", "origin_airport_code", "destination_airport_code",
    "flight_date", "crs_dep_time AS alt_crs_dep_time", "dep_minutes AS alt_dep_minutes",
    "prob_delay_pre_pct AS alt_prob_delay_pct", "risk_pre_departure AS alt_risk",
    "dep_delay AS alt_dep_delay",
)

WINDOW_MINUTES = 180

joined = (
    scored_pool.alias("orig")
    .join(
        candidates.alias("alt"),
        (F.col("orig.origin_airport_code") == F.col("alt.origin_airport_code"))
        & (F.col("orig.destination_airport_code") == F.col("alt.destination_airport_code"))
        & (F.col("orig.flight_date") == F.col("alt.flight_date"))
        & (F.col("orig.fl_number") != F.col("alt.alt_flight"))
        & (F.abs(F.col("orig.dep_minutes") - F.col("alt.alt_dep_minutes")) <= WINDOW_MINUTES),
    )
    .withColumn("improvement_pct",
                F.col("orig.prob_delay_pre_pct") - F.col("alt.alt_prob_delay_pct"))
    .filter(F.col("improvement_pct") >= 10.0)
    .withColumn("risk_bonus",
                F.when(F.col("alt.alt_risk") == "Low", 20)
                 .when(F.col("alt.alt_risk") == "Medium", 10).otherwise(0))
    .withColumn("recommendation_score", F.col("improvement_pct") + F.col("risk_bonus"))
)

window = Window.partitionBy(
    "orig.airline_code", "orig.fl_number", "orig.flight_date"
).orderBy(F.col("recommendation_score").desc())

recommendations = (
    joined.withColumn("recommendation_rank", F.row_number().over(window))
    .filter(F.col("recommendation_rank") <= 5)
    .select(
        F.col("orig.airline_name").alias("original_airline"),
        F.col("orig.fl_number").alias("original_flight"),
        F.col("orig.origin_airport_code").alias("origin"),
        F.col("orig.destination_airport_code").alias("destination"),
        F.col("orig.flight_date").alias("flight_date"),
        F.col("orig.prob_delay_pre_pct").alias("original_delay_prob"),
        F.col("orig.dep_delay").alias("original_dep_delay"),
        F.col("alt.alt_airline").alias("alternative_airline"),
        F.col("alt.alt_airline_code").alias("alternative_airline_code"),
        F.col("alt.alt_flight").alias("alternative_flight"),
        F.col("alt.alt_prob_delay_pct").alias("alternative_delay_prob"),
        F.col("alt.alt_risk").alias("alternative_risk_level"),
        F.col("alt.alt_dep_delay").alias("alternative_dep_delay"),
        "improvement_pct", "recommendation_score", "recommendation_rank",
        F.current_timestamp().alias("recommendation_timestamp"),
    )
)

(
    recommendations.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(config.ALTERNATIVES)
)
print(f"Wrote {recommendations.count():,} recommendations -> {config.ALTERNATIVES}")
display(recommendations.orderBy(F.desc("recommendation_score")).limit(10))


Wrote 0 recommendations -> workspace.flights.alternative_flight_recommendations


original_airline,original_flight,origin,destination,flight_date,original_delay_prob,original_dep_delay,alternative_airline,alternative_airline_code,alternative_flight,alternative_delay_prob,alternative_risk_level,alternative_dep_delay,improvement_pct,recommendation_score,recommendation_rank,recommendation_timestamp


## Loop closed

Every model above was loaded by `models:/catalog.schema.name@champion` — no run ID, no
version pinned in code — and scored at a threshold read off the model itself. Retraining
promotes a new champion and this notebook picks it up unchanged.

That is the defect the original project died on, demonstrated working end to end.


In [0]:
summary = spark.sql(f"""
    SELECT scoring_date,
           COUNT(*)                                   AS flights_scored,
           SUM(predicted_delayed_pre)                 AS flagged_pre_departure,
           SUM(predicted_delayed_in)                  AS flagged_in_flight,
           ROUND(AVG(prob_delay_pre_pct), 2)          AS avg_pre_pct,
           ROUND(AVG(prob_delay_in_pct), 2)           AS avg_in_pct
    FROM {config.PREDICTIONS}
    GROUP BY scoring_date ORDER BY scoring_date DESC
""")
display(summary)

print(f"pre-departure champion : {pre_name} v{pre_version} @ {PRE_THRESHOLD:.2f}")
print(f"in-flight champion     : {in_name} v{in_version} @ {IN_THRESHOLD:.2f}")
